In [1]:
# 数据显示格式设置
import pandas as pd
pd.set_option('display.expand_frame_repr',False)
pd.set_option('display.max_columns',None)

In [2]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def generate_nightmare_ecommerce_data(n=1000):
    np.random.seed(42)
    
    # 基础时间戳（带有混乱的时区偏移）
    base_dates = [datetime(2025, 1, 1) + timedelta(days=np.random.randint(0, 365)) for _ in range(n)]
    
    data = {
        'transaction_id': [f'TXN_{10000 + i}' for i in range(n)],
        'order_time_utc': base_dates,
        # 挑战点1：混合地区，不同国家的税率和货币单位完全不同
        'region': np.random.choice(['EU', 'US', 'SEA', 'CN', 'LATAM'], n),
        # 挑战点2：脏数据 - 包含负数金额、字符串格式的数字、以及缺失值
        'raw_amount': [np.random.choice([round(np.random.uniform(10, 5000), 2), -99, "1,200.50", None]) for _ in range(n)],
        # 挑战点3：多种结算货币
        'currency': np.random.choice(['USD', 'EUR', 'JPY', 'GBP', 'IDR'], n),
        # 挑战点4：物流状态与支付状态的逻辑冲突（例如：未支付但已发货）
        'payment_status': np.random.choice(['Paid', 'Pending', 'Refunded', 'Failed'], n, p=[0.7, 0.15, 0.1, 0.05]),
        'shipping_status': np.random.choice(['Delivered', 'In Transit', 'Cancelled'], n)
    }
    
    df = pd.DataFrame(data)
    
    # 故意制造一些逻辑地雷：支付失败但显示已送达
    df.loc[df['payment_status'] == 'Failed', 'shipping_status'] = 'Delivered'
    
    return df

# 生成数据
df_challenge = generate_nightmare_ecommerce_data(100)
print("地雷阵已部署，前5行预览：")
print(df_challenge.head())
print(df_challenge.info())

地雷阵已部署，前5行预览：
  transaction_id order_time_utc region raw_amount currency payment_status shipping_status
0      TXN_10000     2025-04-13     CN        -99      GBP           Paid       Cancelled
1      TXN_10001     2025-12-15    SEA       None      JPY           Paid       Delivered
2      TXN_10002     2025-09-28    SEA        -99      GBP           Paid      In Transit
3      TXN_10003     2025-04-17     EU       None      IDR           Paid       Cancelled
4      TXN_10004     2025-03-13    SEA       None      EUR           Paid       Delivered
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (total 7 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   transaction_id   100 non-null    object        
 1   order_time_utc   100 non-null    datetime64[ns]
 2   region           100 non-null    object        
 3   raw_amount       69 non-null     object        
 4   currency         100

In [12]:
# 创建副本进行清洗
df_cleaned = df_challenge.copy()

# 单独创建一列，标记所有数据为'Normal'
df_cleaned['issue_flag'] = 'Normal'

# 1.标记出那些“付了钱但没金额”的灵异订单
df_cleaned.loc[df_cleaned['raw_amount'].isna() & (df_cleaned['payment_status']=='Paid'),'issue_flag'] = 'Missing Amount but Paid'

# 2. 统计一下各种异常标签的数量
print("数据异常分布统计：")
print(df_cleaned['issue_flag'].value_counts())

# 3. 看看那几个“灵异订单”
print("\n付了钱但没金额的典型样本：")
print(df_cleaned[df_cleaned['issue_flag'] == 'Missing Amount but Paid'].head())
print(df_cleaned['raw_amount'])

数据异常分布统计：
issue_flag
Normal                     80
Missing Amount but Paid    20
Name: count, dtype: int64

付了钱但没金额的典型样本：
   transaction_id order_time_utc region raw_amount currency payment_status shipping_status               issue_flag
1       TXN_10001     2025-12-15    SEA       None      JPY           Paid       Delivered  Missing Amount but Paid
3       TXN_10003     2025-04-17     EU       None      IDR           Paid       Cancelled  Missing Amount but Paid
4       TXN_10004     2025-03-13    SEA       None      EUR           Paid       Delivered  Missing Amount but Paid
12      TXN_10012     2025-04-10     CN       None      EUR           Paid      In Transit  Missing Amount but Paid
15      TXN_10015     2025-05-11     US       None      IDR           Paid       Delivered  Missing Amount but Paid
0          -99
1         None
2          -99
3         None
4         None
        ...   
95        None
96        None
97        None
98    1,200.50
99      435.88
Name: raw_amount,

In [6]:
# 1. 物理类型分布
print("【1. 物理类型统计】")
print(df_cleaned['raw_amount'].apply(lambda x: type(x).__name__).value_counts())

【1. 物理类型统计】
raw_amount
NoneType    31
str         27
int         24
float       18
Name: count, dtype: int64


In [8]:
# 2. 采样看看具体的非数字内容（如果有的话）
print("\n【2. 非数字内容采样】")
non_numeric = df_cleaned[df_cleaned['raw_amount'].apply(lambda x: not isinstance(x, (int, float)))]
print(non_numeric['raw_amount'].unique()) # 只看前5个不同的脏数据


【2. 非数字内容采样】
[None '1,200.50']


In [10]:
# 3. 统计空值比例
null_count = df_cleaned['raw_amount'].isna().sum()
print(f"\n【3. 缺失值统计】: {null_count} 个 (占比 {null_count/len(df_cleaned)*100:.1f}%)")


【3. 缺失值统计】: 31 个 (占比 31.0%)


In [13]:
# .str 只能处理字符串，所以先用 .astype(str) 保证万无一失
temp_amount = df_cleaned['raw_amount'].astype(str).str.replace(',','')

In [22]:
df_cleaned['raw_amount'] = pd.to_numeric(df_cleaned['raw_amount'],errors='coerce')
df_cleaned.loc[df_cleaned['raw_amount']<0,'raw_amount'] = np.nan
print(df_cleaned['raw_amount'].dtype)

float64


In [23]:
# 汇率转换
# 假设这是我们从银行接口拿到的实时汇率表
exchange_rates = {
    'USD': 1.0,
    'EUR': 1.08,   # 1欧元 = 1.08美元
    'JPY': 0.0067, # 1日元 = 0.0067美元
    'GBP': 1.27,   # 1英镑 = 1.27美元
    'IDR': 0.000064 # 1印尼盾 = 0.000064美元
}
# 1. 创建一个“汇率列”
# map() 会根据 currency 列的值（如 'GBP'），去 exchange_rates 字典里找对应的数字（如 1.27）
df_cleaned['rate'] = df_cleaned['currency'].map(exchange_rates)

# 2. 直接相乘，得到统一的美元金额
df_cleaned['amount_usd'] = df_cleaned['raw_amount'] * df_cleaned['rate']

# 3. 看看前 10 行的转换结果
print(df_cleaned[['currency', 'raw_amount', 'rate', 'amount_usd']].head(10))

  currency  raw_amount      rate   amount_usd
0      GBP         NaN  1.270000          NaN
1      JPY         NaN  0.006700          NaN
2      GBP         NaN  1.270000          NaN
3      IDR         NaN  0.000064          NaN
4      EUR         NaN  1.080000          NaN
5      GBP         NaN  1.270000          NaN
6      EUR     4017.67  1.080000  4339.083600
7      JPY         NaN  0.006700          NaN
8      USD         NaN  1.000000          NaN
9      JPY      460.55  0.006700     3.085685


In [24]:
# 1. 计算除了空值以外的平均客单价（美元）
avg_amount = df_cleaned['amount_usd'].mean()
print(f"当前有数据的订单平均金额: ${avg_amount:.2f}")

# 2. 把所有的 NaN 填补为平均值
# 这样你的总销售额统计就不会因为缺失值而大幅缩水了
df_cleaned['amount_usd_filled'] = df_cleaned['amount_usd'].fillna(avg_amount)

# 3. 看看填补后的效果
print("\n填补后的数据预览：")
print(df_cleaned[['currency', 'raw_amount', 'amount_usd', 'amount_usd_filled']].head(10))

当前有数据的订单平均金额: $2307.59

填补后的数据预览：
  currency  raw_amount   amount_usd  amount_usd_filled
0      GBP         NaN          NaN        2307.588054
1      JPY         NaN          NaN        2307.588054
2      GBP         NaN          NaN        2307.588054
3      IDR         NaN          NaN        2307.588054
4      EUR         NaN          NaN        2307.588054
5      GBP         NaN          NaN        2307.588054
6      EUR     4017.67  4339.083600        4339.083600
7      JPY         NaN          NaN        2307.588054
8      USD         NaN          NaN        2307.588054
9      JPY      460.55     3.085685           3.085685


C:\Users\ryd19\AppData\Local\Temp\ipykernel_15556\575633969.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df_cleaned['amount_usd_filled'] = df_cleaned['amount_usd'].fillna(avg_amount)
